---

# Part 3 — Label efficiency: how much labelled data does this actually need?

Addresses items 4, 5, 7 and 9 of the 13 Nov MoM, and closes item 10 by putting the models from
the literature review into the roster.

**What this measures.** Parts 1 and 2 train on every label we have. This part asks a different
question: if only *p* percent of the training set were labelled, how good would the model be?
We sweep p from 1% to 100% and plot accuracy against it.

**Why it has to be simulated.** PlantVillage is 100% labelled, so "what percentage of our data is
labelled" has no measurable answer — there is nothing unlabelled in it. The honest framing is to
*hold out* labels: take a stratified p% of the training split as the labelled set, and treat the
remaining images either as absent (supervised baseline) or as unlabelled data to exploit
(pseudo-labelling, Step 15). Both are run here, so the MoM's open question is answered with
numbers rather than a choice made in advance.

**Why the subsets are nested.** The 1% set is a subset of the 2% set, which is a subset of the 5%
set, and so on. Without nesting, a dip in the curve could be a bad draw rather than a real effect.
Each class is shuffled once per seed and the fractions take prefixes of that shuffle, so every
class keeps at least one labelled example even at 1%.

**Why this is affordable.** Stage 1 trains only a head on features the frozen backbone has already
cached, so one point on the curve costs seconds, not minutes. The entire probe sweep —
5 backbones x 7 fractions x up to 3 seeds — runs in a few minutes once the features exist.
Only the fine-tuned points cost real GPU time, so those are taken at three fractions rather than
seven.

**What the 25% slice in Part 2 was, and was not.** That slice existed to rank backbones cheaply
under equal conditions. It is not a point on this curve: every model there still saw a
fully-labelled subset, and the axis being varied was compute, not supervision.

## Prerequisite

Run Part 2 first in the same session — this part reuses `FeatureExtractor`, `ClassifierHead`,
`make_loaders`, `cache_features`, `finetune` and `score` from Step 6b onward. The first cell
checks for them and stops with a clear message if they are missing.

In [ ]:
# =====================================================================
# STEP 12 | PART-3 CONFIG + REFRESHED MODEL ROSTER
# =====================================================================
_needed = ["FeatureExtractor", "ClassifierHead", "make_loaders", "cache_features",
           "finetune", "score", "CFG2", "CLASSES", "N_CLASSES", "TRAIN_TARGETS", "DEVICE"]
_missing = [n for n in _needed if n not in globals()]
if _missing:
    raise RuntimeError("Run Part 2 (Steps 6a onward) in this session first - missing: "
                       + ", ".join(_missing))

import itertools


class CFG3:
    # ---- the sweep ---------------------------------------------------
    fractions    = [0.01, 0.02, 0.05, 0.10, 0.25, 0.50, 1.00]
    seeds_small  = [42, 43, 44]     # fractions <= small_cut get repeated draws
    seeds_large  = [42]
    small_cut    = 0.10

    # ---- stage 1 (probe on cached features) --------------------------
    probe_epochs = 30               # more than Part 2: the small subsets need them
    probe_lr     = 3e-3
    probe_wd     = 1e-4

    # ---- stage 2 (fine-tuning: expensive, fewer points) --------------
    ft_fractions = [0.01, 0.10, 0.50]
    ft_epochs    = 5

    # ---- semi-supervised extension (Step 15) -------------------------
    pseudo_fractions = [0.01, 0.05, 0.10]
    pseudo_threshold = 0.95
    pseudo_rounds    = 2

    results_csv  = CFG2.audit_root / "13_label_efficiency.csv"
    ft_csv       = CFG2.audit_root / "14_label_efficiency_finetuned.csv"
    pseudo_csv   = CFG2.audit_root / "15_pseudo_labelling.csv"


# ---------------------------------------------------------------------
# Roster, revised after the literature review (see LITERATURE.md).
# MoM asks for 4-5 models: two base, two or three recent.
# ---------------------------------------------------------------------
ROSTER = [
    # --- base models --------------------------------------------------
    BackboneSpec("resnet50",    "resnet50",
                 ("layer4",), "base / CNN residual (2015)"),
    BackboneSpec("effnet_b0",   "efficientnet_b0",
                 ("blocks.5", "blocks.6", "conv_head", "bn2"), "base / CNN scaled (2019)"),
    # --- recent models --------------------------------------------------
    BackboneSpec("mnetv4_s",    "mobilenetv4_conv_small.e2400_r224_in1k",
                 ("blocks.3", "blocks.4", "conv_head", "norm_head"), "new / CNN edge (2024)"),
    BackboneSpec("dinov3_vits", "vit_small_patch16_dinov3.lvd1689m",
                 ("blocks.8", "blocks.9", "blocks.10", "blocks.11", "norm."),
                 "new / self-supervised ViT (2025)"),
    BackboneSpec("dinov3_cnxt", "convnext_tiny.dinov3_lvd1689m",
                 ("stages.3", "head."), "new / self-supervised CNN (2025)"),
]

# Kept available but off by default. convnext_tiny is the ImageNet-supervised twin of
# dinov3_cnxt - running both isolates the pretraining recipe from the architecture.
EXTRAS = [
    BackboneSpec("convnext_t",  "convnext_tiny",
                 ("stages.3", "norm_pre", "head."), "extra / supervised twin of dinov3_cnxt"),
    BackboneSpec("mnetv3_s",    "mobilenetv3_small_100",
                 ("blocks.4", "blocks.5", "conv_head"), "extra / superseded by mnetv4_s"),
    BackboneSpec("deit_s",      "deit_small_patch16_224",
                 ("blocks.8", "blocks.9", "blocks.10", "blocks.11", "norm."),
                 "extra / supervised ViT reference"),
]

RUN_EXTRAS = False
ACTIVE = ROSTER + (EXTRAS if RUN_EXTRAS else [])
SPEC.update({b.key: b for b in ROSTER + EXTRAS})

# DINOv3 weights are self-supervised; a fine-tune at the usual rate disturbs them more than it
# helps, so the trunk LR is halved for those two.
LR_MULT = {"dinov3_vits": 0.5, "dinov3_cnxt": 0.5}

rows = []
for s in ACTIVE:
    ex = FeatureExtractor(s, pretrained=False)
    ex.freeze(); n_un = ex.unfreeze_top()
    tot = sum(p.numel() for p in ex.trunk.parameters())
    rows.append({"backbone": s.key, "timm_id": s.timm_id, "role": s.family,
                 "params_M": round(tot / 1e6, 2), "feat_dim": ex.feat_dim,
                 "unfrozen_%": round(100 * n_un / tot, 1),
                 "trunk_lr_mult": LR_MULT.get(s.key, 1.0)})
    del ex

display(pd.DataFrame(rows).set_index("backbone"))
print("active backbones:", len(ACTIVE), "| all unfreeze keys matched")
print("fractions:", CFG3.fractions)
print("labelled images at each fraction:",
      {f: int(round(f * len(TRAIN_TARGETS))) for f in CFG3.fractions})

## Step 13 — Nested labelled subsets, and the probe sweep

For each seed, every class is shuffled once; the fraction *p* takes the first `round(p x n_c)`
images of each class, with a floor of one. That makes the subsets nested across fractions and
stratified by construction, and it guarantees no class disappears at 1%.

The sweep then trains the shared head on the **cached frozen features** of the labelled subset
only, and evaluates on the full validation and test splits. The backbone never sees a gradient
here, so a point costs seconds and the whole grid is affordable at three seeds for the small
fractions, where the variance between draws is largest and a single run would be misleading.

Reported per point: validation and test macro-F1, mean and standard deviation across seeds, plus
the number of labelled images and the mean labelled images per class — the number an annotation
budget is actually expressed in.

In [ ]:
# =====================================================================
# STEP 13 | NESTED LABELLED SUBSETS + STAGE-1 SWEEP
# =====================================================================

def nested_label_subsets(targets, fractions, seed):
    """Nested, stratified, at least one image per class.
    subsets[f1] is a subset of subsets[f2] whenever f1 < f2."""
    rng = np.random.default_rng(seed)
    per_class = []
    for c in range(N_CLASSES):
        idx = np.where(targets == c)[0]
        rng.shuffle(idx)
        per_class.append(idx)
    out = {}
    for f in sorted(fractions):
        take = [idx[:max(1, int(round(f * len(idx))))] for idx in per_class]
        out[f] = np.sort(np.concatenate(take))
    return out


# ---- verify the construction before spending any GPU time -----------
_s = nested_label_subsets(TRAIN_TARGETS, CFG3.fractions, 42)
for a, b in zip(sorted(CFG3.fractions)[:-1], sorted(CFG3.fractions)[1:]):
    assert set(_s[a]).issubset(set(_s[b])), "subsets are not nested at " + str(a) + " -> " + str(b)
for f in CFG3.fractions:
    assert len(np.unique(TRAIN_TARGETS[_s[f]])) == N_CLASSES, "a class vanished at " + str(f)
print("nested + stratified + all classes present at every fraction")
display(pd.DataFrame([{"fraction": f, "labelled": len(_s[f]),
                       "per_class_mean": round(len(_s[f]) / N_CLASSES, 1)}
                      for f in CFG3.fractions]).set_index("fraction"))


def fit_probe(X, Y, idx, Xva, Yva, Xte, Yte, dim, epochs=None, seed=SEED):
    """Train the shared head on cached features of X[idx]. Returns val/test metrics."""
    epochs = epochs or CFG3.probe_epochs
    torch.manual_seed(seed)
    head = ClassifierHead(dim, N_CLASSES, CFG2.head_hidden, CFG2.head_dropout).to(DEVICE)
    Xtr = X[idx]
    Ytr = Y[idx]
    cnt = torch.bincount(Ytr, minlength=N_CLASSES).float()
    w = (cnt.sum() / (N_CLASSES * cnt.clamp(min=1))).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=CFG2.label_smooth)
    opt = torch.optim.AdamW(head.parameters(), lr=CFG3.probe_lr, weight_decay=CFG3.probe_wd)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    n = len(Xtr)
    bs = int(min(CFG2.probe_bs, max(8, n // 4)))

    @torch.no_grad()
    def predict(Xs):
        head.eval()
        return torch.cat([head(Xs[i:i + 4096].float()).argmax(1)
                          for i in range(0, len(Xs), 4096)]).cpu().numpy()

    best_f1, best_state = -1.0, None
    for ep in range(epochs):
        head.train()
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, bs):
            b = perm[i:i + bs]
            if len(b) < 2:          # BatchNorm1d needs at least two samples
                continue
            opt.zero_grad(set_to_none=True)
            crit(head(Xtr[b].float()), Ytr[b]).backward()
            opt.step()
        sch.step()
        f1 = score(Yva.cpu().numpy(), predict(Xva))["macro_f1"]
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}

    head.load_state_dict(best_state)
    return (score(Yva.cpu().numpy(), predict(Xva)),
            score(Yte.cpu().numpy(), predict(Xte)),
            head)


# ---- the sweep -------------------------------------------------------
sweep_rows = []
for spec in ACTIVE:
    print("[" + spec.key + "]  " + spec.timm_id)
    paths = cache_features(spec)                      # no-op if already cached
    d = {s: torch.load(paths[s], weights_only=True) for s in ("train", "val", "test")}
    dim = int(d["train"]["dim"])
    X, Y   = d["train"]["f"].to(DEVICE), d["train"]["y"].to(DEVICE)
    Xva, Yva = d["val"]["f"].to(DEVICE),  d["val"]["y"].to(DEVICE)
    Xte, Yte = d["test"]["f"].to(DEVICE), d["test"]["y"].to(DEVICE)

    for frac in CFG3.fractions:
        seeds = CFG3.seeds_small if frac <= CFG3.small_cut else CFG3.seeds_large
        for sd in seeds:
            subs = nested_label_subsets(TRAIN_TARGETS, CFG3.fractions, sd)
            idx = torch.as_tensor(subs[frac], device=DEVICE, dtype=torch.long)
            vm, tm, _ = fit_probe(X, Y, idx, Xva, Yva, Xte, Yte, dim, seed=sd)
            sweep_rows.append({"backbone": spec.key, "role": spec.family, "stage": "probe",
                               "fraction": frac, "seed": sd,
                               "n_labelled": int(len(idx)),
                               "per_class": round(len(idx) / N_CLASSES, 1),
                               "val_f1": round(vm["macro_f1"], 4),
                               "test_f1": round(tm["macro_f1"], 4),
                               "test_acc": round(tm["acc"], 4)})
        last = [r for r in sweep_rows if r["backbone"] == spec.key and r["fraction"] == frac]
        print("   " + format(frac, ".0%").rjust(5) + "  n=" + str(last[0]["n_labelled"]).rjust(6) +
              "  test macro-F1 " + format(np.mean([r["test_f1"] for r in last]), ".4f") +
              (" +/- " + format(np.std([r["test_f1"] for r in last]), ".4f") if len(last) > 1 else ""))

    del X, Y, Xva, Yva, Xte, Yte, d
    torch.cuda.empty_cache()

sweep = pd.DataFrame(sweep_rows)
sweep.to_csv(CFG3.results_csv, index=False)
print("\nwritten:", CFG3.results_csv, "|", len(sweep), "runs")

## Step 14 — The same sweep with fine-tuning

The probe measures what frozen ImageNet (or DINOv3) features already separate. This measures what
happens when the top of the backbone is allowed to adapt to the labelled subset it is given.

Three fractions rather than seven, one seed each, because each point is a full fine-tuning run.
The head is warm-started from the probe trained on the *same* labelled subset, so the comparison
between the two stages is like-for-like.

The DINOv3 trunks get half the usual learning rate. Their weights are self-supervised rather than
tuned for classification, and the usual rate disturbs more than it repairs at these subset sizes;
`LR_MULT` in Step 12 holds the factor.

Expect roughly 25-40 minutes per backbone on a T4 for the three fractions together — 1% and 10%
are quick, 50% dominates. Results append to disk after every run, so a disconnect costs one point.

In [ ]:
# =====================================================================
# STEP 14 | STAGE-2 SWEEP - FINE-TUNE AT SELECTED LABEL FRACTIONS
# =====================================================================
from contextlib import contextmanager

@contextmanager
def trunk_lr(mult):
    """Part 2's finetune() reads CFG2.ft_backbone_lr; scale it for this run only."""
    base = CFG2.ft_backbone_lr
    CFG2.ft_backbone_lr = base * mult
    try:
        yield
    finally:
        CFG2.ft_backbone_lr = base


FT_PATH = CFG3.ft_csv
ft_rows = pd.read_csv(FT_PATH).to_dict("records") if FT_PATH.exists() else []
done = {(r["backbone"], r["fraction"]) for r in ft_rows}
SEED_FT = CFG3.seeds_large[0]

for spec in ACTIVE:
    pending = [f for f in CFG3.ft_fractions if (spec.key, f) not in done]
    if not pending:
        print("[" + spec.key + "] all fractions done, skipping")
        continue
    print("[" + spec.key + "]  " + spec.timm_id)

    paths = cache_features(spec)
    d = {s: torch.load(paths[s], weights_only=True) for s in ("train", "val", "test")}
    dim = int(d["train"]["dim"])
    X, Y     = d["train"]["f"].to(DEVICE), d["train"]["y"].to(DEVICE)
    Xva, Yva = d["val"]["f"].to(DEVICE),   d["val"]["y"].to(DEVICE)
    Xte, Yte = d["test"]["f"].to(DEVICE),  d["test"]["y"].to(DEVICE)
    subs = nested_label_subsets(TRAIN_TARGETS, CFG3.fractions, SEED_FT)

    for frac in pending:
        sub = subs[frac]
        print("  fraction " + format(frac, ".0%") + "  (" + str(len(sub)) + " labelled)")

        # warm start: probe on exactly this labelled subset
        idx = torch.as_tensor(sub, device=DEVICE, dtype=torch.long)
        _, probe_test, head = fit_probe(X, Y, idx, Xva, Yva, Xte, Yte, dim, seed=SEED_FT)
        warm = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}

        tag = "ft_p" + str(int(round(frac * 100)))
        with trunk_lr(LR_MULT.get(spec.key, 1.0)):
            res, _, _ = finetune(spec, sub, CFG3.ft_epochs, tag,
                                 warm_head=warm, verbose=True)

        ft_rows.append({"backbone": spec.key, "role": spec.family, "fraction": frac,
                        "seed": SEED_FT, "n_labelled": int(len(sub)),
                        "probe_test_f1": round(probe_test["macro_f1"], 4),
                        "ft_test_f1": res["ft_test_f1"],
                        "ft_test_acc": res["ft_test_acc"],
                        "lift": round(res["ft_test_f1"] - probe_test["macro_f1"], 4),
                        "train_min": res["train_min"]})
        pd.DataFrame(ft_rows).to_csv(FT_PATH, index=False)
        print("   probe " + format(probe_test["macro_f1"], ".4f") +
              " -> fine-tuned " + format(res["ft_test_f1"], ".4f") +
              "  (" + str(res["train_min"]) + " min)\n")

    del X, Y, Xva, Yva, Xte, Yte, d
    torch.cuda.empty_cache()

ft = pd.DataFrame(ft_rows)
display(ft)
print("written:", FT_PATH)

## Step 15 — Using the unlabelled remainder: pseudo-labelling

The MoM leaves one question open: when p% of the data is labelled, does the other (100-p)% get
thrown away or used? Step 13 throws it away. This step uses it, so the choice is settled by
measurement rather than assumption.

The method is deliberately the simplest one that works, because it runs entirely on cached
features and so costs seconds per round:

1. Train the head on the labelled subset.
2. Predict the unlabelled remainder; keep every image whose softmax confidence clears 0.95.
3. Add those with their predicted labels and retrain from scratch.
4. Repeat once more.

Two rounds, because the third rarely moves anything and each round compounds whatever errors the
previous one introduced. Confidence thresholding is a weak filter — a confidently wrong prediction
is exactly the one that gets promoted — so the interesting outcome is not only whether F1 rises
but whether it *falls*, which would say the frozen features are not reliable enough at that
labelling level to bootstrap from. Either result is a finding worth reporting.

This is the supervised-vs-semi-supervised comparison at matched label budgets. Run at 1%, 5% and
10%, where the unlabelled pool is large enough for the question to matter.

In [ ]:
# =====================================================================
# STEP 15 | SEMI-SUPERVISED: PSEUDO-LABELLING ON CACHED FEATURES
# =====================================================================

@torch.no_grad()
def predict_proba(head, Xs):
    head.eval()
    return torch.cat([torch.softmax(head(Xs[i:i + 4096].float()), dim=1)
                      for i in range(0, len(Xs), 4096)])


pseudo_rows = []
for spec in ACTIVE:
    print("[" + spec.key + "]")
    paths = cache_features(spec)
    d = {s: torch.load(paths[s], weights_only=True) for s in ("train", "val", "test")}
    dim = int(d["train"]["dim"])
    X, Y     = d["train"]["f"].to(DEVICE), d["train"]["y"].to(DEVICE)
    Xva, Yva = d["val"]["f"].to(DEVICE),   d["val"]["y"].to(DEVICE)
    Xte, Yte = d["test"]["f"].to(DEVICE),  d["test"]["y"].to(DEVICE)
    subs = nested_label_subsets(TRAIN_TARGETS, CFG3.fractions, SEED_FT)

    for frac in CFG3.pseudo_fractions:
        lab = np.sort(subs[frac])
        unl = np.setdiff1d(np.arange(len(TRAIN_TARGETS)), lab)
        idx_l = torch.as_tensor(lab, device=DEVICE, dtype=torch.long)
        idx_u = torch.as_tensor(unl, device=DEVICE, dtype=torch.long)

        _, base_test, head = fit_probe(X, Y, idx_l, Xva, Yva, Xte, Yte, dim, seed=SEED_FT)
        cur_f1 = base_test["macro_f1"]
        Xaug, Yaug, n_pseudo, correct = X[idx_l], Y[idx_l], 0, None

        for rnd in range(CFG3.pseudo_rounds):
            probs = predict_proba(head, X[idx_u])
            conf, pl = probs.max(1)
            keep = conf >= CFG3.pseudo_threshold
            n_pseudo = int(keep.sum())
            if n_pseudo == 0:
                print("   " + format(frac, ".0%") + " round " + str(rnd + 1) +
                      ": nothing above threshold, stopping")
                break
            # how many pseudo-labels are actually right - diagnostic only, never used in training
            correct = float((pl[keep] == Y[idx_u][keep]).float().mean())
            Xaug = torch.cat([X[idx_l], X[idx_u][keep]])
            Yaug = torch.cat([Y[idx_l], pl[keep]])
            all_idx = torch.arange(len(Xaug), device=DEVICE)
            _, tm, head = fit_probe(Xaug, Yaug, all_idx, Xva, Yva, Xte, Yte, dim, seed=SEED_FT)
            cur_f1 = tm["macro_f1"]
            print("   " + format(frac, ".0%") + " round " + str(rnd + 1) +
                  ": +" + format(n_pseudo, ",") + " pseudo-labels (" +
                  format(correct, ".1%") + " correct)  test macro-F1 " + format(cur_f1, ".4f"))

        pseudo_rows.append({"backbone": spec.key, "fraction": frac,
                            "n_labelled": int(len(lab)), "n_unlabelled": int(len(unl)),
                            "supervised_f1": round(base_test["macro_f1"], 4),
                            "pseudo_f1": round(cur_f1, 4),
                            "delta": round(cur_f1 - base_test["macro_f1"], 4),
                            "n_pseudo_final": n_pseudo,
                            "pseudo_correct": round(correct, 4) if correct is not None else None})

    del X, Y, Xva, Yva, Xte, Yte, d
    torch.cuda.empty_cache()

pseudo = pd.DataFrame(pseudo_rows)
pseudo.to_csv(CFG3.pseudo_csv, index=False)
display(pseudo)
print("written:", CFG3.pseudo_csv)

## Step 16 — Results: the label-efficiency curve

Three outputs.

**The curve.** Test macro-F1 against labelled fraction on a log x-axis, one line per backbone,
shaded by standard deviation across seeds where more than one was run. The fine-tuned points from
Step 14 are overlaid as markers so the gap between frozen and adapted features is visible at each
label budget.

**The budget table.** For each backbone, the smallest fraction whose mean test macro-F1 reaches
95% of its own full-label score, converted into labelled images and labelled images per class.
This is the number to put in front of a supervisor or a grower: *this model needs roughly N
labelled leaves per class to get within 5% of its ceiling.* It is also the number that decides
whether a deployment is affordable, which raw accuracy does not.

**Supervised vs pseudo-labelled**, at matched budgets, from Step 15.

Read the curve for shape, not just height. A backbone that starts high and stays flat is one whose
pretrained features already solve the task; a backbone that starts low and climbs steeply is one
that needs supervision to be useful. The DINOv3 models are in the roster to test exactly that, and
their ImageNet-supervised twin is one flag away in `EXTRAS` if the comparison needs isolating.

In [ ]:
# =====================================================================
# STEP 16 | CURVES, ANNOTATION-BUDGET TABLE, FIGURES
# =====================================================================
agg = (sweep.groupby(["backbone", "role", "fraction"])
            .agg(test_f1=("test_f1", "mean"), sd=("test_f1", "std"),
                 n_labelled=("n_labelled", "first"), runs=("test_f1", "size"))
            .reset_index())
agg["sd"] = agg["sd"].fillna(0.0)
agg["per_class"] = (agg["n_labelled"] / N_CLASSES).round(1)
display(agg.pivot(index="fraction", columns="backbone", values="test_f1").round(4))

# ---- the curve -------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 6))
for key, g in agg.groupby("backbone"):
    g = g.sort_values("fraction")
    ax.plot(g.fraction, g.test_f1, marker="o", ms=5, label=key)
    ax.fill_between(g.fraction, g.test_f1 - g.sd, g.test_f1 + g.sd, alpha=0.15)
if len(ft):
    for key, g in ft.groupby("backbone"):
        ax.scatter(g.fraction, g.ft_test_f1, marker="^", s=90, zorder=5,
                   edgecolor="black", linewidth=0.6)
ax.set_xscale("log")
ax.set_xticks(CFG3.fractions)
ax.set_xticklabels([format(f, ".0%") for f in CFG3.fractions])
ax.set_xlabel("labelled fraction of the training split (log scale)")
ax.set_ylabel("test macro-F1")
ax.set_title("Label efficiency - circles: frozen features, triangles: fine-tuned")
ax.grid(alpha=0.25)
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig(CFG2.audit_root / "label_efficiency_curve.png", dpi=150)
plt.show()

# ---- annotation budget ----------------------------------------------
budget = []
for key, g in agg.groupby("backbone"):
    g = g.sort_values("fraction")
    ceiling = float(g[g.fraction == 1.0].test_f1.iloc[0])
    target = 0.95 * ceiling
    hit = g[g.test_f1 >= target]
    row = {"backbone": key, "full_label_f1": round(ceiling, 4),
           "target_95pct": round(target, 4)}
    if len(hit):
        h = hit.iloc[0]
        row.update({"reached_at": format(h.fraction, ".0%"),
                    "labelled_images": int(h.n_labelled),
                    "per_class": h.per_class,
                    "f1_there": round(float(h.test_f1), 4)})
    else:
        row.update({"reached_at": "not within swept range", "labelled_images": None,
                    "per_class": None, "f1_there": None})
    budget.append(row)
budget = pd.DataFrame(budget).sort_values("full_label_f1", ascending=False)
budget.to_csv(CFG2.audit_root / "16_annotation_budget.csv", index=False)
print("\nLabels needed to reach 95% of each model's own full-label macro-F1:")
display(budget)

# ---- frozen vs fine-tuned vs pseudo-labelled -------------------------
if len(ft):
    cmp = ft[["backbone", "fraction", "n_labelled", "probe_test_f1", "ft_test_f1", "lift"]]
    print("\nFrozen features vs fine-tuning, at matched label budgets:")
    display(cmp.sort_values(["backbone", "fraction"]))

if len(pseudo):
    print("\nSupervised vs pseudo-labelled, at matched label budgets:")
    display(pseudo[["backbone", "fraction", "n_labelled", "supervised_f1",
                    "pseudo_f1", "delta", "pseudo_correct"]])
    fig, ax = plt.subplots(figsize=(8, 4.5))
    w = 0.35
    x = np.arange(len(pseudo))
    ax.bar(x - w / 2, pseudo.supervised_f1, w, label="supervised only", color="#90a4ae")
    ax.bar(x + w / 2, pseudo.pseudo_f1, w, label="+ pseudo-labelling", color="#2e7d32")
    ax.set_xticks(x)
    ax.set_xticklabels([r.backbone + "\n" + format(r.fraction, ".0%")
                        for r in pseudo.itertuples()], fontsize=7)
    ax.set_ylabel("test macro-F1")
    ax.set_title("Using the unlabelled remainder")
    ax.legend(); ax.grid(alpha=0.25, axis="y")
    plt.tight_layout()
    plt.savefig(CFG2.audit_root / "pseudo_labelling.png", dpi=150)
    plt.show()

summary = {
    "fractions": CFG3.fractions,
    "seeds_small": CFG3.seeds_small, "seeds_large": CFG3.seeds_large,
    "backbones": [s.key for s in ACTIVE],
    "probe_runs": int(len(sweep)),
    "finetune_runs": int(len(ft)),
    "pseudo_runs": int(len(pseudo)),
    "annotation_budget": budget.to_dict("records"),
}
(CFG2.audit_root / "16_label_efficiency_summary.json").write_text(json.dumps(summary, indent=2))
print("\nwritten:", CFG2.audit_root / "16_label_efficiency_summary.json")

## Outputs of Part 3

| Artefact | What it is |
|---|---|
| `audit/13_label_efficiency.csv` | every probe run: backbone x fraction x seed, val and test macro-F1 |
| `audit/14_label_efficiency_finetuned.csv` | fine-tuned points, with the probe score at the same budget beside each |
| `audit/15_pseudo_labelling.csv` | supervised vs pseudo-labelled at matched budgets, including how many pseudo-labels were actually correct |
| `audit/16_annotation_budget.csv` | labelled images per class needed to reach 95% of each model's ceiling |
| `audit/16_label_efficiency_summary.json` | the whole sweep configuration and headline result in one file |
| `label_efficiency_curve.png`, `pseudo_labelling.png` | figures for the paper |

## Reading the result honestly

Three things to check before claiming anything from the curve.

**Variance at 1%.** At 1% each class has roughly 33 labelled images, and the spread across seeds
will be wide. That is why the small fractions run three seeds. If the standard deviation at 1% is
comparable to the gap between two backbones, the two are not distinguishable at that budget and
the write-up should say so rather than ranking them.

**The pseudo-label trap.** `pseudo_correct` in `15_pseudo_labelling.csv` reports what fraction of
the promoted labels were actually right. It is a diagnostic, computed against ground truth that a
genuine deployment would not have, and it is never used in training. If F1 rises while
`pseudo_correct` is low, the gain is coming from somewhere other than correct labels and deserves
suspicion.

**The ceiling is per model.** The budget table measures each backbone against *its own* full-label
score, not against the best model. A weak backbone can look label-efficient simply by having a low
ceiling, so read the budget column next to `full_label_f1`, never on its own.

## Still outstanding

Grad-CAM++ explainability — block 8 of the architecture, and the one item of the original design
that neither Part 2 nor Part 3 touches. Joseph et al. (2026) in `LITERATURE.md` pair few-shot
learning with Grad-CAM/++ on exactly this problem and are the natural reference for it. Start from
the `top_confusions` list in `12_final_report.json`.